# Squad Advanced Stats — 2025/26 UCL Season

Advanced metrics (xG, xA, ball recoveries, tackles, ratings) pulled from Sofascore's data API — tracks the same style of stats StatsBomb is known for (StatsBomb's own free data doesn't cover recent seasons).

**Note:** Mbeumo, Mainoo and Maguire show no rows — Manchester United weren't in the 2025/26 Champions League, so they have no UCL data for that season. Rodri, Gordon and Grimaldo have since transferred clubs (to Barcelona, Barcelona and Atlético Madrid respectively) — stats below are from their 2025/26 season at their old clubs (Man City, Newcastle, Leverkusen).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

In [ ]:
data = [
    # Player, Position, Apps, Mins, Goals, Assists, xG, xA, Rating, KeyPasses, BallRecovery, Tackles, Interceptions, CleanSheet, Saves, Yellow, Red
    ("Isak",        "Forward",    7,  285, 0,  0, 0.93, 0.57, 6.39,  2,   4,  4,  1, 0,  0, 0, 0),
    ("Kane",        "Forward",   13, 1039, 14, 2, 9.32, 2.48, 7.93, 16,  27,  4,  5, 1,  0, 0, 0),
    ("Raphinha",    "Midfielder", 7,  544, 3,  2, 2.36, 1.78, 7.24, 21,  17, 10,  3, 0,  0, 0, 0),
    ("Rodri",       "Midfielder", 5,  347, 0,  0, 0.39, 0.64, 7.10,  7,  21,  7,  4, 0,  0, 1, 1),
    ("Vitinha",     "Midfielder",17, 1544, 6,  1, 3.36, 2.80, 7.58, 26,  95, 25, 17, 5,  0, 0, 0),
    ("Gordon",      "Midfielder",12,  770, 10, 2, 8.55, 1.08, 7.50, 11,  27,  7,  3, 1,  0, 1, 0),
    ("Gabriel",     "Defender",  12, 1002, 1,  1, 0.92, 0.15, 7.18,  2,  23, 14,  5, 6,  0, 1, 0),
    ("Grimaldo",    "Defender",  12, 1079, 4,  3, 2.35, 2.52, 7.33, 29,  64, 38, 13, 4,  0, 2, 0),
    ("Nuno Mendes", "Defender",  17, 1403, 2,  2, 1.37, 2.35, 7.09, 23, 102, 27, 18, 2,  0, 4, 0),
    ("Davies",      "Defender",   8,  213, 0,  2, 0.02, 0.42, 6.79,  6,  17,  6,  2, 0,  0, 0, 0),
    ("Neuer",       "Goalkeeper",11,  990, 0,  0, None, 0.02, 7.06,  0,  85,  0,  0, 2, 37, 2, 0),
    ("Chevalier",   "Goalkeeper", 6,  540, 0,  0, None, 0.00, 6.67,  0,  37,  0,  0, 1,  9, 0, 0),
]
cols = ["Player", "Position", "Apps", "Mins", "Goals", "Assists", "xG", "xA", "Rating",
        "KeyPasses", "BallRecovery", "Tackles", "Interceptions", "CleanSheets", "Saves", "Yellow", "Red"]

df = pd.DataFrame(data, columns=cols)

missing = pd.DataFrame([
    ("Mbeumo", "Forward"), ("Mainoo", "Midfielder"), ("Maguire", "Defender")
], columns=["Player", "Position"])

df

## 1. Full stats table by position

In [ ]:
pos_order = ["Goalkeeper", "Defender", "Midfielder", "Forward"]
df["Position"] = pd.Categorical(df["Position"], categories=pos_order, ordered=True)
df_sorted = df.sort_values(["Position", "Rating"], ascending=[True, False]).set_index(["Position", "Player"])
df_sorted

## 2. Goal output vs. underlying quality (xG)

Comparing actual goals to expected goals shows who's overperforming (clinical finishing) vs. underperforming (unlucky, or due a return).

In [ ]:
attackers = df[df["Position"].isin(["Forward", "Midfielder"])].sort_values("xG", ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(attackers))
width = 0.35
ax.bar(x - width/2, attackers["Goals"], width, label="Actual goals", color="#e63946")
ax.bar(x + width/2, attackers["xG"], width, label="xG (expected)", color="#a8dadc")
ax.set_xticks(x)
ax.set_xticklabels(attackers["Player"], rotation=30, ha='right')
ax.set_ylabel("Goals")
ax.set_title("Goals vs. xG — 2025/26 UCL")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Sofascore rating by player, grouped by position

In [ ]:
colors = {"Goalkeeper": "#f2a900", "Defender": "#3d7ea6", "Midfielder": "#4caf50", "Forward": "#e63946"}
df_r = df.sort_values(["Position", "Rating"])

fig, ax = plt.subplots(figsize=(8, 6))
bar_colors = [colors[p] for p in df_r["Position"]]
ax.barh(df_r["Player"], df_r["Rating"], color=bar_colors)
ax.set_xlim(6, 8.2)
ax.set_xlabel("Average Sofascore rating")
ax.set_title("Squad average match rating — 2025/26 UCL")

from matplotlib.patches import Patch
handles = [Patch(color=c, label=p) for p, c in colors.items()]
ax.legend(handles=handles, loc='lower right')
plt.tight_layout()
plt.show()

## 4. Ball recoveries — directly relevant to Fantasy points (every 3 = +1)

In [ ]:
df["RecoveryPoints"] = (df["BallRecovery"] // 3).astype(int)
rec = df.sort_values("BallRecovery", ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = [colors[p] for p in rec["Position"]]
ax.bar(rec["Player"], rec["BallRecovery"], color=bar_colors)
ax.set_ylabel("Ball recoveries (2025/26 UCL)")
ax.set_title("Ball recoveries by player — fuels Fantasy's 'every 3 recoveries = +1' rule")
ax.tick_params(axis='x', rotation=45)
for i, (v, p) in enumerate(zip(rec["BallRecovery"], rec["RecoveryPoints"])):
    ax.text(i, v + 1, f"+{p}pt", ha='center', fontsize=8)
plt.tight_layout()
plt.show()

## 5. Player radar — pick a player to inspect

A StatsBomb-style radar chart normalized against your squad's own range for each stat.

In [ ]:
def plot_radar(player_name, metrics=("Goals", "Assists", "xG", "xA", "KeyPasses", "BallRecovery", "Tackles")):
    row = df[df["Player"] == player_name].iloc[0]
    valid_metrics = [m for m in metrics if pd.notna(row[m])]
    values = []
    for m in valid_metrics:
        col = df[m].dropna()
        lo, hi = col.min(), col.max()
        norm = 0.5 if hi == lo else (row[m] - lo) / (hi - lo)
        values.append(norm)
    values += values[:1]
    angles = np.linspace(0, 2*np.pi, len(valid_metrics), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(5, 5), subplot_kw=dict(polar=True))
    ax.plot(angles, values, color=colors[row["Position"]], linewidth=2)
    ax.fill(angles, values, color=colors[row["Position"]], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(valid_metrics)
    ax.set_yticklabels([])
    ax.set_title(f"{player_name} — relative to squad ({row['Position']})", y=1.1)
    plt.tight_layout()
    plt.show()

plot_radar("Vitinha")

In [ ]:
# Try any player in the squad, e.g.:
plot_radar("Grimaldo")

## 6. Missing data

These squad players have no 2025/26 UCL stats because Manchester United did not qualify for that season's Champions League:

In [ ]:
missing

## 7. League-wide top players by stat (2025/26 UCL)

Full-competition leaderboards, with your squad highlighted in red wherever they appear.

In [ ]:
squad_highlight = {"Kane", "Gordon", "Grimaldo", "Vitinha"}

leaders = {
    "Goals": [("Mbappé", 15), ("Kane", 14), ("Kvaratskhelia", 10), ("Alvarez", 10), ("Gordon", 10),
               ("Dembélé", 8), ("Haaland", 8), ("Luis Díaz", 7)],
    "xG": [("Mbappé", 10.79), ("Kane", 9.32), ("Gordon", 8.55), ("Alvarez", 8.30), ("Vinícius Jr", 8.09),
            ("Osimhen", 7.77), ("Haaland", 6.82), ("Dembélé", 6.55)],
    "Assists": [("Kvaratskhelia", 6), ("Olise", 6), ("Hakimi", 6), ("Vinícius Jr", 5), ("Gnabry", 5),
                 ("Yamal", 4), ("Alvarez", 4), ("Szoboszlai", 4)],
    "xA": [("Olise", 4.92), ("Salah", 4.33), ("Yamal", 4.31), ("Luis Díaz", 4.21), ("Alvarez", 3.97),
            ("Wirtz", 3.80), ("Szoboszlai", 3.41), ("Vinícius Jr", 3.39)],
    "Tackles": [("Khalaili", 34), ("Araújo", 43), ("Van de Perre", 30), ("Mac Allister", 29), ("Brown", 28),
                 ("Grimaldo", 38), ("Dahl", 28), ("Bensebaini", 27)],
    "Rating": [("Yamal", 8.08), ("Mbappé", 8.05), ("Kane", 7.93), ("Kvaratskhelia", 7.71), ("Olise", 7.61),
                ("Vitinha", 7.58), ("Alvarez", 7.55), ("Szoboszlai", 7.53)],
    "Clean sheets": [("Raya", 9), ("Vicario", 6), ("Pope", 4), ("Courtois", 4), ("Alisson", 4),
                       ("Sommer", 4), ("Safonov", 4), ("Çakır", 3)],
    "Saves": [("Haikin", 67), ("Courtois", 53), ("Rulli", 36), ("Kochalski", 43), ("Çakır", 50),
               ("Kovář", 31), ("Pope", 29), ("Köhn", 29)],
}

fig, axes = plt.subplots(4, 2, figsize=(13, 18))
for ax, (stat, rows) in zip(axes.flat, leaders.items()):
    names_, vals = zip(*rows)
    bar_colors = ["#e63946" if n in squad_highlight else "#cccccc" for n in names_]
    ax.barh(names_[::-1], vals[::-1], color=bar_colors[::-1])
    ax.set_title(stat)

fig.suptitle("2025/26 UCL leaders by stat — red = your squad", y=1.0, fontsize=14)
plt.tight_layout()
plt.show()

**Your squad's standout league rankings (2025/26 UCL):**
- **Kane** — 2nd in goals (14) and xG (9.32)
- **Gordon** — 5th in goals (10, tied), 3rd in xG (8.55)
- **Grimaldo** — 2nd in tackles (38)
- **Vitinha** — 6th in average rating (7.58)

None of your other players (Isak, Raphinha, Rodri, Gabriel, Nuno Mendes, Davies, Neuer, Chevalier) cracked the top 8 in these particular categories last season — still useful players, just not league-leading by these specific stats.

## 8. Domestic league performance — 2025/26

Same players, their actual domestic league (Premier League, Bundesliga, LaLiga, Ligue 1) — a much bigger sample size than UCL alone, and includes Mbeumo, Mainoo and Maguire who had no UCL minutes.

In [ ]:
league_data = [
    # Player, Position, League, Apps, Mins, Goals, Assists, xG, xA, Rating, KeyPasses, BallRecovery, Tackles, Interceptions, CleanSheet, Saves, Yellow, Red
    ("Isak",        "Forward",    "Premier League", 14, 714,  3,  1, 2.65,  0.10, 6.57, 2,  13,  7,  0,  0,  0, 0, 0),
    ("Mbeumo",      "Forward",    "Premier League", 33, 2621, 11, 3, 12.19, 5.04, 6.84, 47, 91, 25,  8,  0,  0, 4, 0),
    ("Kane",        "Forward",    "Bundesliga",      31, 2382, 36, 5, 26.87, 4.27, 7.84, 40, 77, 19,  8,  4,  0, 1, 0),
    ("Raphinha",    "Midfielder", "LaLiga",          22, 1388, 13, 3, 10.36, 5.81, 7.51, 43, 47, 15,  5,  3,  0, 5, 0),
    ("Mainoo",      "Midfielder", "Premier League",  28, 1670, 1,  2, 0.48,  1.28, 6.88, 22, 95, 36, 20,  6,  0, 2, 0),
    ("Rodri",       "Midfielder", "Premier League",  21, 1513, 1,  0, 1.13,  2.59, 7.43, 25, 107,41, 14,  3,  0, 3, 0),
    ("Vitinha",     "Midfielder", "Ligue 1",         29, 2121, 1,  7, 1.06,  3.74, 7.58, 34, 137,24, 27, 10,  0, 2, 0),
    ("Gordon",      "Midfielder", "Premier League",  26, 1817, 6,  2, 8.83,  3.37, 7.05, 26, 57, 19,  3,  2,  0, 3, 1),
    ("Gabriel",     "Defender",   "Premier League",  32, 2751, 3,  4, 2.93,  1.82, 7.27, 7,  64, 38, 23, 17,  0, 4, 0),
    ("Grimaldo",    "Defender",   "Bundesliga",      29, 2524, 8,  8, 6.65,  10.13,7.31, 70, 127,49, 19,  5,  0, 5, 0),
    ("Maguire",     "Defender",   "Premier League",  23, 1667, 1,  2, 1.11,  0.52, 6.88, 4,  62, 22, 13,  5,  0, 3, 1),
    ("Nuno Mendes", "Defender",   "Ligue 1",         20, 1251, 4,  5, 4.33,  3.63, 7.24, 30, 82, 25, 13,  5,  0, 1, 0),
    ("Davies",      "Defender",   "Bundesliga",      13, 534,  1,  3, 0.35,  1.63, 6.82, 9,  51,  9,  3,  0,  0, 1, 0),
    ("Neuer",       "Goalkeeper", "Bundesliga",      22, 1860, 0,  0, None,  0.04, 6.89, 0,  182, 0,  0,  6, 31, 0, 0),
    ("Chevalier",   "Goalkeeper", "Ligue 1",         17, 1530, 0,  0, None,  0.00, 6.95, 0,  108, 0,  0,  9, 33, 0, 0),
]
lcols = ["Player", "Position", "League", "Apps", "Mins", "Goals", "Assists", "xG", "xA", "Rating",
         "KeyPasses", "BallRecovery", "Tackles", "Interceptions", "CleanSheets", "Saves", "Yellow", "Red"]

league_df = pd.DataFrame(league_data, columns=lcols)
league_df["Position"] = pd.Categorical(league_df["Position"], categories=pos_order, ordered=True)
league_df.sort_values(["Position", "Rating"], ascending=[True, False]).set_index(["Position", "Player"])

In [ ]:
attackers_l = league_df[league_df["Position"].isin(["Forward", "Midfielder"])].sort_values("xG", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(attackers_l))
width = 0.35
ax.bar(x - width/2, attackers_l["Goals"], width, label="Actual goals", color="#e63946")
ax.bar(x + width/2, attackers_l["xG"], width, label="xG (expected)", color="#a8dadc")
ax.set_xticks(x)
ax.set_xticklabels(attackers_l["Player"] + "\n(" + attackers_l["League"] + ")", rotation=30, ha='right', fontsize=8)
ax.set_ylabel("Goals")
ax.set_title("Goals vs. xG — 2025/26 domestic league")
ax.legend()
plt.tight_layout()
plt.show()

## 9. League-wide top players by stat — each player's own domestic league

Leaderboards for the Premier League, Bundesliga, LaLiga and Ligue 1 (2025/26), with your squad highlighted in red.

In [ ]:
squad_highlight_domestic = {"Isak", "Mbeumo", "Kane", "Raphinha", "Mainoo", "Rodri", "Vitinha", "Gordon",
                             "Gabriel", "Grimaldo", "Maguire", "Nuno Mendes", "Davies", "Neuer", "Chevalier"}

league_leaders = {
    "Premier League — Goals": [("Haaland", 27), ("Igor Thiago", 22), ("Semenyo", 17), ("Watkins", 16),
                                 ("João Pedro", 15), ("Gibbs-White", 15)],
    "Premier League — Assists": [("Bruno Fernandes", 21), ("Cherki", 12), ("Bowen", 11), ("Haaland", 8),
                                   ("Garner", 7), ("Szoboszlai", 7)],
    "Bundesliga — Goals": [("Kane", 36), ("Undav", 19), ("Guirassy", 17), ("Schick", 16), ("Olise", 15), ("Luis Díaz", 15)],
    "Bundesliga — xA": [("Olise", 17.23), ("Raum", 12.39), ("Schmid", 10.44), ("Kimmich", 10.24),
                          ("Grimaldo", 10.13), ("Diomande", 10.12)],
    "LaLiga — Goals": [("Mbappé", 25), ("Muriqi", 23), ("Budimir", 17), ("Yamal", 16), ("Vinícius Jr", 16), ("Ferran Torres", 16)],
    "LaLiga — Rating": [("Yamal", 7.93), ("Mbappé", 7.56), ("Pedri", 7.56), ("Vinícius Jr", 7.50),
                          ("Dituro", 7.36), ("Joan García", 7.34)],
    "Ligue 1 — Assists": [("Ajorque", 9), ("Thomasson", 9), ("Vitinha", 7), ("Dembélé", 7), ("Greenwood", 7), ("Endrick", 7)],
    "Ligue 1 — Rating": [("Vitinha", 7.58), ("Kebbal", 7.39), ("Greenwood", 7.37), ("Koffi", 7.35),
                           ("Beraldo", 7.25), ("Mandi", 7.24)],
}

fig, axes = plt.subplots(4, 2, figsize=(13, 18))
for ax, (title, rows) in zip(axes.flat, league_leaders.items()):
    names_, vals = zip(*rows)
    bar_colors = ["#e63946" if n in squad_highlight_domestic else "#cccccc" for n in names_]
    ax.barh(names_[::-1], vals[::-1], color=bar_colors[::-1])
    ax.set_title(title, fontsize=10)

fig.suptitle("2025/26 domestic league leaders — red = your squad", y=1.0, fontsize=14)
plt.tight_layout()
plt.show()

**Your squad's standout domestic rankings (2025/26):**
- **Kane** — #1 goalscorer in the Bundesliga (36 goals, way clear) and #1 xG (26.87)
- **Grimaldo** — 5th in Bundesliga xA (10.13), among midfielders/defenders
- **Vitinha** — 3rd in Ligue 1 assists (7, tied), #1 in Ligue 1 average rating (7.58)
- Isak, Mbeumo, Raphinha, Mainoo, Rodri, Gordon, Gabriel, Maguire, Nuno Mendes, Davies, Neuer, Chevalier didn't crack their league's top 6 in these categories — solid contributors, not table-topping specialists by these specific stats.

Kane's Bundesliga season in particular is exceptional — nearly double the goals of the next-best scorer.

In [ ]:
current_data = [
    # Player, Position, League, Apps, Mins, Goals, Assists, xG, xA, Rating, KeyPasses, BallRecovery, Tackles, CleanSheet, Yellow, Red
    ("Isak",        "Forward",    "Premier League", 3, 244, 3, 0, 2.29, 0.06, 7.63, 2,  2,  1, 0, 0, 0),
    ("Mbeumo",      "Forward",    "Premier League", 3, 270, 2, 0, 2.31, 0.47, 7.07, 5, 10,  3, 0, 1, 0),
    ("Kane",        "Forward",    "Bundesliga",      2, 129, 0, 0, 0.30, 0.68, 6.65, 3,  2,  0, 0, 0, 0),
    ("Raphinha",    "Midfielder", "LaLiga",          4, 319, 6, 1, 5.40, 0.88, 8.58, 10, 5,  1, 1, 0, 0),
    ("Mainoo",      "Midfielder", "Premier League",  3, 187, 0, 1, 0.21, 0.35, 7.37, 2, 21,  5, 0, 0, 0),
    ("Rodri",       "Midfielder", "LaLiga",          3, 115, 0, 0, 0.08, 0.11, 7.00, 2,  7,  3, 0, 1, 0),
    ("Vitinha",     "Midfielder", "Ligue 1",         3, 249, 1, 1, 0.32, 1.26, 8.17, 5, 14,  3, 0, 0, 0),
    ("Gordon",      "Midfielder", "LaLiga",          4, 244, 0, 4, 1.21, 0.90, 7.18, 6,  7,  2, 0, 0, 0),
    ("Gabriel",     "Defender",   "Premier League",  3, 270, 0, 0, 0.49, 0.05, 6.83, 0, 11,  2, 2, 1, 0),
    ("Grimaldo",    "Defender",   "LaLiga",          3, 270, 0, 0, 0.12, 0.15, 6.50, 0,  9,  9, 0, 0, 0),
    ("Maguire",     "Defender",   "Premier League",  3, 270, 0, 0, 0.08, 0.42, 6.80, 2, 11,  1, 0, 1, 0),
    ("Davies",      "Defender",   "Bundesliga",      2, 164, 0, 0, None,  0.79, 7.45, 5, 13,  4, 1, 0, 0),
    ("Neuer",       "Goalkeeper", "Bundesliga",      2, 180, 0, 0, None,  0.20, 7.45, 1,  9,  0, 1, 0, 0),
]
ccols = ["Player", "Position", "League", "Apps", "Mins", "Goals", "Assists", "xG", "xA", "Rating",
         "KeyPasses", "BallRecovery", "Tackles", "CleanSheets", "Yellow", "Red"]

current_df = pd.DataFrame(current_data, columns=ccols)
current_df["Position"] = pd.Categorical(current_df["Position"], categories=pos_order, ordered=True)

current_missing = pd.DataFrame([("Nuno Mendes", "Defender"), ("Chevalier", "Goalkeeper")], columns=["Player", "Position"])

current_df.sort_values(["Position", "Rating"], ascending=[True, False]).set_index(["Position", "Player"])

## 11. Three-way comparison: previous UCL vs. previous domestic vs. this season so far

Rating and goals+assists per 90, side by side. This season's numbers are from a tiny sample (2-4 games) — early signal only, not yet reliable form.

In [ ]:
def rating_lookup(frame, player):
    row = frame[frame["Player"] == player]
    return row["Rating"].iloc[0] if len(row) else np.nan

squad_order = ["Isak", "Mbeumo", "Kane", "Raphinha", "Mainoo", "Rodri", "Vitinha", "Gordon",
               "Gabriel", "Grimaldo", "Maguire", "Nuno Mendes", "Davies", "Neuer", "Chevalier"]

compare = pd.DataFrame({
    "Player": squad_order,
    "Prev UCL (25/26)": [rating_lookup(df, p) for p in squad_order],
    "Prev domestic (25/26)": [rating_lookup(league_df, p) for p in squad_order],
    "Current domestic (26/27, early)": [rating_lookup(current_df, p) for p in squad_order],
})

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(compare))
width = 0.26
ax.bar(x - width, compare["Prev UCL (25/26)"], width, label="Prev UCL 25/26", color="#264653")
ax.bar(x, compare["Prev domestic (25/26)"], width, label="Prev domestic 25/26", color="#2a9d8f")
ax.bar(x + width, compare["Current domestic (26/27, early)"], width, label="Current 26/27 (early)", color="#e76f51")
ax.set_xticks(x)
ax.set_xticklabels(compare["Player"], rotation=45, ha='right')
ax.set_ylabel("Average rating")
ax.set_ylim(6, 9)
ax.set_title("Rating across three samples — previous UCL, previous domestic, current season so far")
ax.legend()
plt.tight_layout()
plt.show()

compare

**Early reads from this season (small sample, treat cautiously):**
- **Raphinha** — 8.58 rating, 6 goals in 4 LaLiga games, well above both his UCL and domestic form last season
- **Vitinha** — 8.17 rating in Ligue 1, up from 7.58 last season
- **Kane** — quieter start (6.65, 0 goals in 2 Bundesliga games) after a 36-goal season — small sample, not yet a trend
- **Grimaldo** — 6.50 at Atlético so far, below his 7.31 at Leverkusen last season — new club adjustment

## Next steps

- Once 2026/27 matchdays accumulate, re-pull each player's stats with `unique-tournament/7/season/96518/statistics/overall` (UCL) — currently returns no data since too few matches have been played this season.
- Source: Sofascore's public statistics API (`sofascore.com/api/v1/player/{id}/unique-tournament/{t}/season/{s}/statistics/overall`), read via browser fetch since it blocks non-browser requests.

## Next steps

- Once 2026/27 matchdays accumulate, re-pull each player's stats with `unique-tournament/7/season/<2026-27 season id>/statistics/overall` and append a season column to compare form.
- Source: Sofascore's public statistics API (`sofascore.com/api/v1/player/{id}/unique-tournament/7/season/{seasonId}/statistics/overall`), read via browser fetch since it blocks non-browser requests.